In [ ]:
import cv2
import numpy as np

def cartonator(image_path, output_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load image: {image_path}")
        return


    img_balanced = cv2.convertScaleAbs(img, alpha=1.2, beta=10)


    hsv = cv2.cvtColor(img_balanced, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    s = cv2.multiply(s, 1.4) # 채도 적용
    v = cv2.multiply(v, 1.1) # 밝기 적용
    img_vivid = cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2BGR)


    sharpen_kernel = np.array([[-1, -1, -1],
                                [-1, 9, -1], 
                                [-1, -1, -1]])
    img_sharp = cv2.filter2D(img_vivid, -1, sharpen_kernel)


    color_simplified = cv2.bilateralFilter(img_sharp, d=7, sigmaColor=50, sigmaSpace=50)

    gray = cv2.cvtColor(img_balanced, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 100, 200)


    mask = cv2.bitwise_not(edges)
    mask_color = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    result = cv2.bitwise_and(color_simplified, mask_color)

    combined = np.hstack((img, result))


    cv2.imshow("catonator", combined)
    cv2.imwrite(output_path, combined)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

cartonator('rap.jpg', 'rap_theory_final.jpg')